In [ ]:
import torch
import json
import numpy as np
import re

In [ ]:
def compute_mae_rmse(y_true, y_pred):
    """
    计算 MAE 和 RMSE 指标。

    参数:
    - y_true: 真实值 (numpy array or list)
    - y_pred: 预测值 (numpy array or list)

    返回:
    - MAE: 平均绝对误差
    - RMSE: 均方根误差
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return mae, rmse

In [ ]:
## 数据集
with open('/root/autodl-tmp/GPT-api/LEVIR-MCI-dataset/LevirCCcaptions-v2.json', 'r') as f:
    datajson = json.load(f)

datajson_t = [x['change_counts'] for x in datajson if x['filepath'] == 'test']
with open(r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/20250206131/test_epochbest.json', 'r') as f:
    data = json.load(f)

In [ ]:
results = []
text = 'there are no changes in roads or buildings'
text1 = 'no new roads or buildings'
for item in data:
    result = {}
    cap = item['caption']
    result['image_id'] = item['image_id']
    if text in cap:       
        result['change_counts'] = {"road": 0, "building": 0}
        results.append(result)
        continue
    if text1 in cap:
        result['change_counts'] = {"road": 0, "building": 0}
        results.append(result)
        continue
    change_count = {}
    caps = cap.split('new')
    r_text = caps[0]
    b_text = caps[1]
    if 'no' in r_text:
        change_count['road']=0
    else:
        change_count['road'] = int(re.findall(r'\d+', r_text)[0])
    if 'no' in b_text:
        change_count['building'] = 0
    else:
        change_count['building'] = int(re.findall(r'\d+', b_text)[0])
    result['change_counts'] = change_count
    results.append(result)

In [ ]:
datajson = results

In [ ]:
## 道路
roads_t = [x['road'] for x in datajson_t]
roads_p = [x['change_counts']['road'] for x in datajson]
mae, rmse = compute_mae_rmse(roads_t, roads_p)
print('roads')
print('MAE:', mae)
print('RMSE:', rmse)

build_t = [x['building'] for x in datajson_t]
build_p = [x['change_counts']['building'] for x in datajson]
mae, rmse = compute_mae_rmse(build_t, build_p)
print('buildings')
print('MAE:', mae)
print('RMSE:', rmse)